In [1]:
import pandas as pd
import re
import datetime
import logging
import pathlib
import global3d_track as g3d

import matplotlib.pyplot as plt

# datadir = pathlib.Path('/work/bb1153/b382635/data/track_statistics/dcc_frozen_1e4/tropics')
datadir = pathlib.Path(f'/work/bb1153/b382635/data/track_statistics/grl/resubmission')
outdir = pathlib.Path(f'/work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_L2')

outdir.mkdir(parents=1, exist_ok=1)

In [2]:
# import os
# os.system("rm /work/bb1153/b382635/plots/model_paper/grl_resubmission/data/statistics_L2/*")


### load and aggregate key results

#### PROCESS

In [4]:
import new_statistics

In [5]:
files = list(datadir.glob('*'))
N = len(files)
print(N, 'DCCs ready for the making')

5369 DCCs ready for the making


In [6]:
# test file names
regex = r'_(\d+)\.nc$'
df = pd.read_csv('/work/bb1153/b382635/data/final_tracks/dcc_frozen_1e4/tropics/track_stats/dcc_validity.csv', index_col=0)
valid_dccs = df.index[df.valid | df.valid_but_complex_convection]
gfiles = []
misc = []
for f in files:
    try:
        dcc = int(re.search(regex, pathlib.Path(f).name).group(1))
        if dcc in valid_dccs:
            gfiles.append(f)
    except:
        misc.append(f)
print(len(gfiles), 'valid DCCs')

5367 valid DCCs


In [7]:
SIZE = 50

for i in range(1,int(N/SIZE)+2):
    # start batch
    logging.info('batch %s' %i)
    t0 = datetime.datetime.now()
    batch = i
    results_path = outdir / f'b{batch}s{SIZE}.nc'
    data_params = dict(files=gfiles, batch=batch, size=SIZE, regex=regex)

    # exists
    if results_path.exists():
        continue

    # calculate
    stats = g3d.analysis.helpers.load_stats(apply=new_statistics.calculate_all, **data_params)
    stats["system"] = stats["system"].assign_attrs(long_name="unique identifier for a tracked deep convective cloud")
    # stats["core"] = stats["core"].assign_attrs(long_name="unique identifier for a convective core associated with a tracked deep convective cloud")

    logging.info('calculated')

    # save
    stats.to_netcdf(results_path)
    tdelta = datetime.datetime.now() - t0
    logging.info('took %s, saved to %s' %(tdelta, results_path))
    stats.close()

INFO:root:batch 1
INFO:root:batch 2
INFO:root:batch 3
INFO:root:batch 4
INFO:root:batch 5
INFO:root:batch 6
INFO:root:batch 7
INFO:root:batch 8
INFO:root:batch 9
INFO:root:batch 10
INFO:root:batch 11
INFO:root:batch 12
INFO:root:batch 13
INFO:root:batch 14
INFO:root:batch 15
INFO:root:batch 16
INFO:root:batch 17
INFO:root:batch 18
INFO:root:batch 19
INFO:root:batch 20
INFO:root:batch 21
INFO:root:batch 22
INFO:root:batch 23
INFO:root:batch 24
INFO:root:batch 25
INFO:root:batch 26
INFO:root:batch 27
INFO:root:batch 28
INFO:root:batch 29
INFO:root:batch 30
INFO:root:batch 31
INFO:root:batch 32
INFO:root:batch 33
INFO:root:batch 34
INFO:root:batch 35
INFO:root:batch 36
INFO:root:batch 37
INFO:root:batch 38
INFO:root:batch 39
INFO:root:batch 40
INFO:root:batch 41
INFO:root:batch 42
INFO:root:batch 43
INFO:root:batch 44
INFO:root:batch 45
INFO:root:batch 46
INFO:root:batch 47
INFO:root:batch 48
INFO:root:batch 49
INFO:root:batch 50
INFO:root:batch 51
INFO:root:batch 52
INFO:root:batch 53
IN

In [7]:
import xarray as xr
import new_statistics
ds = xr.open_dataset(gfiles[5])
r = new_statistics.calculate_all(ds)

In [8]:
r

<xarray.Dataset> Size: 58kB
Dimensions:            (system: 1, threshold: 9, interp_time: 41, phase: 5)
Coordinates:
  * system             (system) float64 8B 1.52e+04
    lat                float64 8B -12.0
    lon                float64 8B 98.8
    level_full         int32 4B 65
  * threshold          (threshold) float64 72B 0.04245 0.07684 ... 0.8952 1.716
  * interp_time        (interp_time) float64 328B 0.0 0.025 0.05 ... 0.975 1.0
  * phase              (phase) object 40B 'developing' 'mature' ... 'overall'
Data variables: (12/49)
    ABH                float64 8B ...
    lifetime           float64 8B 4.25
    ncores             int64 8B 2
    ocean_fraction     float32 4B ...
    TOI                datetime64[ns] 8B 2021-07-03T06:20:17
    conv_lifetime      float64 8B 2.25
    ...                 ...
    Apred_v1_cont      (phase, system, interp_time) float64 2kB 8.47e+08 ... ...
    Apred_v2           (phase, system, interp_time) float64 2kB 8.47e+08 ... nan
    Apred_v2_cont      (phase, system, interp_time) float64 2kB 8.47e+08 ... ...
    t_max_depth        float64 8B 0.075
    t_max_area         float64 8B 0.575
    lifestages_valid   (system) bool 1B True